# Pydantic

a)

In [1]:
from pydantic import BaseModel

# 1. Define the pydantic model
# we create a class calles "User" that inherits from BaseModel
# we use type hints to define the fields and their types
class User(BaseModel):
    id: int
    name: str

# 2. Test with valid data
# create an instance of the User model with valid data
try:
    valid_user = User(id=123, name="Edwin Lindblom")
    print("Valid user created:\n", valid_user.name)
    print("User ID:\n", valid_user.id)
except Exception as e:
    print("Error with valid data:", e)

Valid user created:
 Edwin Lindblom
User ID:
 123


In [2]:
# test with invalid data
try:
    invalid_user = User(id="not_an_int", name=456)
except Exception as e:
    print("Error with data. Invalid data provided:", e)

Error with data. Invalid data provided: 2 validation errors for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not_an_int', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing
name
  Input should be a valid string [type=string_type, input_value=456, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


b)

In [6]:
from pydantic import BaseModel, EmailStr, Field, ValidationError
from enum import Enum

# Create a enum to define valid pets
class Pet(str, Enum):
    DOG = "dog"
    CAT = "cat"
    FISH = "fish"
    BIRD = "bird"
    RABBIT = "rabbit"
    TURTLE = "turtle"

# Define Person model with validation
class Person(BaseModel):
    name: str 
    age: int = Field(gt=0, lt=150) # age must be between 1 and 149
    email: EmailStr # valid email adress
    favorite_pet: Pet

# Test with valid data
try:
    person1 = Person(name="Michael", age = 45, email="michaelschott@office.com", favorite_pet=Pet.TURTLE)
    print("Person created:\n", person1)
except ValidationError as e:
    print("Error with valid data:", e)


Person created:
 name='Michael' age=45 email='michaelschott@office.com' favorite_pet=<Pet.TURTLE: 'turtle'>


In [17]:
# test with invalid age
try:
    person_invalid_age = Person(name= "Gollum", age = 538, email="ring1@thehobbit.com", favorite_pet= Pet.FISH)
except ValidationError as e:
    print("Error! Invalid age provided(are you really that old?)",(e))


Error! Invalid age provided(are you really that old?) 1 validation error for Person
age
  Input should be less than 150 [type=less_than, input_value=538, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than


In [10]:
# test with invalid email
try:
    person_invalid_email = Person(name="Frodo", age= 27, email="frodo_the_hobbit.com", favorite_pet=Pet.DOG)
except ValidationError as e:
    print("Error! Invalid email provided:(e)")


Error! Invalid email provided:(e)


In [11]:
# test with invalid pet type
try:
    person_invalid_pet = Person(name="Frankenstein", age= 127, email="frank-the-cat@catmail.com", favorite_pet="not dogs")
except ValidationError as e:
    print("Unknown pet type provided:", (e))

Unknown pet type provided: 1 validation error for Person
favorite_pet
  Input should be 'dog', 'cat', 'fish', 'bird', 'rabbit' or 'turtle' [type=enum, input_value='not dogs', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/enum


b)

In [18]:
# import re to validate e-mail with a regular experssion
import re 
from enum import Enum

class Pet(str, Enum):
    DOG = "dog"
    CAT = "cat"
    FISH = "fish"
    BIRD = "bird"
    RABBIT = "rabbit"
    TURTLE = "turtle"

class Person:
    def __init__(self, name: str, age: int, email: str, favorite_pet: Pet):
        # validate name, must be a string
        if not isinstance(name, str):
            raise ValueError("Name must be a string.")
        self.name = name
        # validate age, must be between 1 and 149
        if not (0 < age < 150):
            raise ValueError("Age must be between 1 and 149.")
        self.age = age
        # validate email with a simple regex
        if not re.match(r"[^@]+@[^@]+\.[^@]+", email):
            raise ValueError("Invalid email adress.")
        self.email = email
        # validate favorite_pet, must be one of the defined pets
        if favorite_pet not in Pet:
            raise ValueError(f"Favorite pet must be one of the following: {[pet.value for pet in Pet]}")
        self.favorite_pet = favorite_pet

    def __repr__(self):
        # a helpful method to get a nice printout of the object
        return f"Person(name = {self.name}, age = {self.age}, email = {self.email}, favorite_pet = {self.favorite_pet}"

# test with valid data

try:
    p_valid = Person(name="Alice", age=25, email="alice@wonderland.com", favorite_pet=Pet.RABBIT)
    print("Person created with manual class:\n", p_valid)
except ValueError as e:
    print("Error with valid data:", e)


Person created with manual class:
 Person(name = Alice, age = 25, email = alice@wonderland.com, favorite_pet = Pet.RABBIT


In [32]:
# test our manual error handling with invalid age

try:
    p_invalid_age = Person(name="Bob", age=200, email="bobber@gmail.com", favorite_pet="dog")
except ValueError as e:
    print("Error! unvalid age.")




Error! unvalid age.


In [31]:
# test our manual error handling with invalid pet type

try:
    p_invalid_age = Person(name="Charlie", age=22, email="charlie@gmail.com", favorite_pet="dragon")
except ValueError as e:
    print("Error! Unvalid pet type.")


C:\Users\edwin\AppData\Local\Temp\ipykernel_20120\2952856738.py:28: DeprecationWarning: in 3.12 __contains__ will no longer raise TypeError, but will return True or
False depending on whether the value is a member or the value of a member
  if favorite_pet not in Pet:


TypeError: unsupported operand type(s) for 'in': 'str' and 'EnumType'